In [1]:

import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# -----------------------------------
# 1. LOAD ENVIRONMENT VARIABLES
# -----------------------------------

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

required_vars = {
    "DB_HOST": DB_HOST,
    "DB_PORT": DB_PORT,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_NAME": DB_NAME
}

missing_vars = [
    key for key, value in required_vars.items()
    if not value
]

if missing_vars:
    raise ValueError(
        f"Missing environment variables: {missing_vars}"
    )

# -----------------------------------
# 2. DATABASE CONNECTION
# -----------------------------------

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

# -----------------------------------
# 3. READ CSV
# -----------------------------------

CSV_FILE = "food_delivery_sql_ready.csv"

print("Reading CSV file...")

df = pd.read_csv(CSV_FILE)

print("Dataset shape:", df.shape)

if df["Order_ID"].duplicated().any():
    raise ValueError("Duplicate Order_ID values found!")

# -----------------------------------
# 4. DATA PREPARATION
# -----------------------------------

df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%Y-%m-%d",
    errors="raise"
).dt.date

# Preserve Peak_Hour values as strings
df["Peak_Hour"] = df["Peak_Hour"].map(
    lambda x: None if pd.isna(x) else str(x)
)

# Convert pandas missing values to Python None
df = df.astype(object).where(pd.notna(df), None)

print(
    "Missing Delivery_Rating values:",
    sum(value is None for value in df["Delivery_Rating"])
)

# -----------------------------------
# 5. CHECK MYSQL TABLE
# -----------------------------------

with engine.connect() as connection:

    existing_rows = connection.execute(
        text("SELECT COUNT(*) FROM food_orders")
    ).scalar()

    print("Existing MySQL rows:", existing_rows)

    if existing_rows != 0:
        raise ValueError(
            "Target table is not empty. "
            "Import stopped to prevent duplicates."
        )

# -----------------------------------
# 6. BATCH IMPORT
# -----------------------------------

print("\nStarting MySQL import...")

try:
    with engine.begin() as connection:

        df.to_sql(
            name="food_orders",
            con=connection,
            if_exists="append",
            index=False,
            chunksize=1000,
            method=None
        )

    print("\nImport completed successfully!")

except Exception as error:
    print("\nImport failed:", str(error))
    raise

# -----------------------------------
# 7. VERIFY IMPORT
# -----------------------------------

with engine.connect() as connection:

    total_rows = connection.execute(
        text("SELECT COUNT(*) FROM food_orders")
    ).scalar()

    print("\n========== IMPORT SUMMARY ==========")
    print("Expected rows:", len(df))
    print("Imported rows:", total_rows)
    print("====================================")

    if total_rows != len(df):
        raise ValueError("Imported row count does not match!")

engine.dispose()


Reading CSV file...
Dataset shape: (97011, 25)
Missing Delivery_Rating values: 15999
Existing MySQL rows: 97011


ValueError: Target table is not empty. Import stopped to prevent duplicates.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the project folder
PROJECT_DIR = Path(
    r"C:\TRAINING_CODES\Online_Food_Delivery_Analysis"
)

ENV_FILE = PROJECT_DIR / ".env"

if not ENV_FILE.exists():
    raise FileNotFoundError(
        f".env file not found: {ENV_FILE}"
    )

load_dotenv(dotenv_path=ENV_FILE, override=True)

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

# Verify configuration without exposing password
print("Database host:", DB_HOST)
print("Database name:", DB_NAME)
print("Database user:", DB_USER)
print("Password loaded:", bool(DB_PASSWORD))

if DB_NAME != "food_delivery_analysis":
    raise ValueError("Incorrect database configured!")


Database host: localhost
Database name: food_delivery_analysis
Database user: root
Password loaded: True


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Exact project folder
PROJECT_DIR = Path(
    r"C:\TRAINING_CODES\Online_Food_Delivery_Analysis"
)

ENV_FILE = PROJECT_DIR / ".env"

print("Environment file exists:", ENV_FILE.exists())

if not ENV_FILE.exists():
    raise FileNotFoundError(
        f".env not found at: {ENV_FILE}"
    )

# Load the correct .env file
load_dotenv(ENV_FILE, override=True)

# Read database configuration
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

# Display safe configuration details
print("Host:", DB_HOST)
print("Database:", DB_NAME)
print("User:", DB_USER)
print("Password loaded:", bool(DB_PASSWORD))

if DB_NAME != "food_delivery_analysis":
    raise ValueError(
        "Wrong database name in .env file!"
    )


Environment file exists: True
Host: localhost
Database: food_delivery_analysis
User: root
Password loaded: True


In [ ]:

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

with engine.connect() as connection:

    print(
        "Connected database:",
        connection.execute(
            text("SELECT DATABASE()")
        ).scalar()
    )

    print(
        "Available tables:",
        connection.execute(
            text("SHOW TABLES")
        ).fetchall()
    )

    print(
        "Existing rows:",
        connection.execute(
            text("SELECT COUNT(*) FROM food_orders")
        ).scalar()
    )


Connected database: food_delivery_analysis
Available tables: [('food_orders',)]
Existing rows: 0


In [ ]:

import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# 1. Set project paths
PROJECT_DIR = Path(r"C:\TRAINING_CODES\Online_Food_Delivery_Analysis")
CSV_FILE = PROJECT_DIR / "food_delivery_sql_ready.csv"
ENV_FILE = PROJECT_DIR / ".env"

# 2. Load database credentials
load_dotenv(ENV_FILE, override=True)

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

if DB_NAME != "food_delivery_analysis":
    raise ValueError("Incorrect database selected.")

if not DB_PASSWORD:
    raise ValueError("Database password was not loaded.")

if not CSV_FILE.exists():
    raise FileNotFoundError(f"CSV file not found: {CSV_FILE}")

# 3. Create database connection
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url, pool_pre_ping=True)

# 4. Read CSV
df = pd.read_csv(CSV_FILE)
print("CSV shape:", df.shape)

# 5. Validate expected columns and row count
if len(df) != 97011:
    raise ValueError(f"Unexpected row count: {len(df)}")

if len(df.columns) != 25:
    raise ValueError(f"Unexpected column count: {len(df.columns)}")

# 6. Convert date column
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"], errors="raise"
).dt.date

# 7. Convert missing values to Python None (SQL NULL)
df = df.astype(object).where(pd.notna(df), None)

# 8. Check destination table before import
with engine.connect() as connection:
    current_database = connection.execute(
        text("SELECT DATABASE()")
    ).scalar()

    existing_rows = connection.execute(
        text("SELECT COUNT(*) FROM food_orders")
    ).scalar()

    print("Connected database:", current_database)
    print("Existing rows:", existing_rows)

    if current_database != "food_delivery_analysis":
        raise ValueError("Connected to the wrong database.")

    if existing_rows != 0:
        raise ValueError(
            "Target table is not empty. Import cancelled to avoid duplicates."
        )

# 9. Import in batches
with engine.begin() as connection:
    df.to_sql(
        name="food_orders",
        con=connection,
        if_exists="append",
        index=False,
        chunksize=1000,
        method=None
    )

# 10. Verify imported records
with engine.connect() as connection:
    imported_rows = connection.execute(
        text("SELECT COUNT(*) FROM food_orders")
    ).scalar()

print("\nImport completed!")
print("Expected rows:", len(df))
print("Imported rows:", imported_rows)

if imported_rows != len(df):
    raise ValueError("Row count mismatch. Please investigate.")

engine.dispose()


CSV shape: (97011, 25)
Connected database: food_delivery_analysis
Existing rows: 97011


ValueError: Target table is not empty. Import cancelled to avoid duplicates.